# CS 260C Homework 2

## Deadline: 6/4/2026 at 11:59pm (No Late Submission)

In this assignment you will explore two complementary approaches for learning human preferences from pairwise comparison data:

## Outline
1. **LLM-as-Judge**: prompt Gemini to act as a pairwise evaluator  
2. **Reward Model**: fine-tune a small language model using the Bradley-Terry preference loss

## Instructions
- Make sure you are running this notebook with a **T4 GPU**. Go to `Runtime → Change runtime type → T4 GPU`. You may change to better GPUs if needed.
- Follow the instructions and fill in the code for the sections marked with `# TODO`.
- **DO NOT** modify the checking/grading cells. Modifying these cells is strictly prohibited and will be treated as an academic integrity violation which will result in 0 score for this assignment and escalation to The Office of Student Conduct at UCLA.

## Submission
- **Execution**: Ensure all cells have been run, and outputs are displayed before submission.
- **File Naming**: Save your completed notebook with outputs as `hw2.ipynb`.
- **Upload**: Submit your `hw2.ipynb` file to Gradescope.

Failure to follow these instructions will result in the autograder failing, which will automatically result in 0 points. **No regrading will be done** for submissions with incorrect file names or formats.

## Setup

In [1]:
!pip install -q datasets transformers accelerate google-genai scikit-learn tqdm

In [2]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow. Check Runtime -> Change runtime type.")

GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
import os
import random
import time
from typing import Optional

import numpy as np
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

---
## Part 0 - Data Loading *(provided, do not modify)*

We use [PPE-Human-Preference-V1](https://huggingface.co/datasets/lmarena-ai/PPE-Human-Preference-V1), a collection of ~16 K pairwise battles from Chatbot Arena with real human judgments.

| Column | Description |
|---|---|
| `prompt` | User instruction |
| `model_a` | Model A name |
| `model_b` | Model B name |
| `response_1` | Output from model A |
| `response_2` | Output from model B |
| `winner` | [model_a, model_b, tie, tie (bothbad)] |

Ties are excluded so the task is binary classification. We split 80 / 20 into train and test.

In [4]:
def load_ppe_dataset(test_size: float = 0.2, exclude_ties: bool = True):
    """Load PPE-Human-Preference-V1 and return train / test splits."""
    print("Loading PPE dataset from HuggingFace...")
    raw = load_dataset("lmarena-ai/PPE-Human-Preference-V1", split="test")

    records = []
    for row in raw:
        winner = row["winner"]
        if exclude_ties and winner not in ("model_a", "model_b"):
            continue
        if winner == "model_a":
            label = 0
        elif winner == "model_b":
            label = 1
        else:
            continue
        records.append({
            "prompt":     row["prompt"],
            "response_a": row["response_1"],
            "response_b": row["response_2"],
            "label":      label,
        })

    train_data, test_data = train_test_split(
        records, test_size=test_size, random_state=SEED, shuffle=True
    )
    print(f"  Train: {len(train_data):,}  |  Test: {len(test_data):,}")
    return train_data, test_data


def compute_accuracy(predictions: list, labels: list) -> float:
    """Return fraction of correct predictions."""
    assert len(predictions) == len(labels)
    return sum(p == l for p, l in zip(predictions, labels)) / len(labels)


train_data, test_data = load_ppe_dataset()

Loading PPE dataset from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.71k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/16038 [00:00<?, ? examples/s]

  Train: 8,172  |  Test: 2,044


In [5]:
# Quick look at one example
ex = train_data[0]
print("Prompt:    ", ex["prompt"][:200])
print("\nResponse A:", ex["response_a"][:200])
print("\nResponse B:", ex["response_b"][:200])
print("\nLabel:     ", ex["label"], "(0 = A preferred, 1 = B preferred)")

Prompt:     Please rephrase the following by correcting syntax and grammar errors and make it sound more fluid "Forewords
I started the process by gathering ideas on what I wanted as an experience for this race. 

Response A: Foreword

I began the process by brainstorming ideas to create a memorable experience for this race. I quickly realized that I wanted to design something cool, with several 'wow' moments and a narrati

Response B: Here is a revised version: 

# Foreword

I initiated the process by conceptualizing my desired experience for this race. I soon realized I wanted to create something captivating with numerous awe-insp

Label:      0 (0 = A preferred, 1 = B preferred)


---
## Part 1 - LLM Judge (25 pts)

You will use the **Gemini API** to build a pairwise judge.

**Setup steps:**
1. Sign up at [Google AI Studio](https://aistudio.google.com) and create an API key.
2. In Colab, go to the 🔑 **Secrets** tab (left sidebar) and add a secret named `GEMINI_API_KEY`.
3. Enable notebook access for that secret.

We can use **`gemini-2.5-flash-lite`** — fast and free.

In [6]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# Sanity check
client = genai.Client(api_key=GEMINI_API_KEY)
test = client.models.generate_content(model="gemini-2.5-flash-lite", contents="Reply with the word OK.")
print(test.text)

OK


### 1a - Prompt design (10 pts)

Design a prompt that instructs Gemini to pick the better response.

**Requirements:**
- The model must output **only** the single character `A` or `B`.
- Think about whether to include explicit evaluation criteria.
- *(Bonus)* Consider running each example twice with A/B swapped to reduce position bias.

In [7]:
def build_judge_prompt(prompt: str, response_a: str, response_b: str) -> str:
    """
    Construct the prompt sent to the LLM judge.

    Returns a string that instructs the model to output ONLY 'A' or 'B'.
    """
    return (
        "You are an impartial expert judge comparing two AI assistant responses "
        "to the same user prompt. Decide which response is better overall, "
        "considering helpfulness, relevance, correctness, completeness, and clarity. "
        "Do not let the order of the responses, their length, or the assistant names "
        "bias your decision.\n\n"
        "[User Prompt]\n"
        f"{prompt}\n\n"
        "[Response A]\n"
        f"{response_a}\n\n"
        "[Response B]\n"
        f"{response_b}\n\n"
        "Which response is better? Respond with ONLY a single character: "
        "\"A\" if Response A is better, or \"B\" if Response B is better. "
        "Output no explanation, punctuation, or any other text.\n"
        "Answer:"
    )

In [8]:
# =====================================================
# DO NOT MODIFY
# =====================================================

# Test that build_judge_prompt returns a non-empty string containing both responses
_test_prompt = "What is 2+2?"
_test_a = "The answer is 4."
_test_b = "I believe it equals four."
_result = build_judge_prompt(_test_prompt, _test_a, _test_b)
if not isinstance(_result, str):
    raise ValueError(f"FAILED: build_judge_prompt must return a str, got {type(_result).__name__}.")
if len(_result.strip()) == 0:
    raise ValueError("FAILED: build_judge_prompt returned an empty string.")
if _test_a not in _result:
    raise ValueError("FAILED: build_judge_prompt must include response_a in the returned prompt.")
if _test_b not in _result:
    raise ValueError("FAILED: build_judge_prompt must include response_b in the returned prompt.")
print("Part 1a - build_judge_prompt basic checks passed.")

Part 1a - build_judge_prompt basic checks passed.


### 1b - Response parsing (5 pts)

In [9]:
def parse_judge_response(response_text: str) -> Optional[int]:
    """
    Parse the LLM output into a label.

    Returns:
        0  if A is preferred
        1  if B is preferred
        None if the output cannot be parsed

    Handle edge cases: extra whitespace, lowercase, punctuation, etc.
    """
    if not isinstance(response_text, str):
        return None
    # Strip surrounding whitespace and common trailing/leading punctuation,
    # then normalise case. We require the cleaned answer to be EXACTLY 'A' or 'B'
    # so ambiguous outputs ("AB", "both", "1", ...) fall through to None.
    cleaned = response_text.strip().strip(".,!?;:'\"()[]{}*`-").strip().upper()
    if cleaned == "A":
        return 0
    if cleaned == "B":
        return 1
    return None

In [10]:
# =====================================================
# DO NOT MODIFY
# =====================================================

# Test valid responses -> 0 (A preferred) or 1 (B preferred)
_valid_cases = [
    ("A", 0), ("B", 1),
    (" a ", 0), (" b ", 1),
    ("A.", 0), ("B.", 1),
    (" A\n", 0), (" B\n", 1),
]
for _text, _expected in _valid_cases:
    _got = parse_judge_response(_text)
    if _got != _expected:
        raise ValueError(
            f"FAILED: parse_judge_response({repr(_text)}) returned {_got!r}, expected {_expected}."
        )
# Test unparseable responses -> None
_none_cases = ["C", "neither", "both", "1", "0", "", "AB", "N/A"]
for _text in _none_cases:
    _got = parse_judge_response(_text)
    if _got is not None:
        raise ValueError(
            f"FAILED: parse_judge_response({repr(_text)}) returned {_got!r}, expected None."
        )
print("Part 1b - parse_judge_response edge-case checks passed.")

Part 1b - parse_judge_response edge-case checks passed.


### 1c - Run the judge (10 pts)

We evaluate on 500 examples.

In [11]:
def run_llm_judge(
    test_data: list,
    max_samples: int = 500,
) -> tuple:
    """
    Run the Gemini judge on `max_samples` examples.

    Steps:
      1. Iterate over test_data[:max_samples].
      2. For each example: build_judge_prompt -> Gemini -> parse_judge_response.
      3. Skip examples where parse_judge_response returns None.

    Returns:
        predictions (list[int]), labels (list[int])
    """
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)
    predictions, labels = [], []

    # Deterministic, single-token answer keeps the judge cheap and consistent.
    gen_config = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=5,
    )

    for ex in tqdm(test_data[:max_samples], desc="LLM judge"):
        judge_prompt = build_judge_prompt(
            ex["prompt"], ex["response_a"], ex["response_b"]
        )

        text = None
        for attempt in range(3):  # simple retry for transient/rate-limit errors
            try:
                resp = client.models.generate_content(
                    model="gemini-2.5-flash-lite",
                    contents=judge_prompt,
                    config=gen_config,
                )
                text = resp.text
                break
            except Exception:
                time.sleep(2 ** attempt)

        pred = parse_judge_response(text) if text is not None else None
        if pred is None:
            continue

        predictions.append(pred)
        labels.append(ex["label"])

    return predictions, labels

In [12]:
judge_preds, judge_labels = run_llm_judge(test_data, max_samples=500)
judge_acc = compute_accuracy(judge_preds, judge_labels)
print(f"LLM Judge Accuracy: {judge_acc:.4f}  ({len(judge_preds)} examples)")

LLM judge:   0%|          | 0/500 [00:00<?, ?it/s]

LLM Judge Accuracy: 0.8261  (23 examples)


In [13]:
# =====================================================
# DO NOT MODIFY
# =====================================================

if not isinstance(judge_preds, list):
    raise ValueError(f"FAILED: judge_preds must be a list, got {type(judge_preds).__name__}.")
if not isinstance(judge_labels, list):
    raise ValueError(f"FAILED: judge_labels must be a list, got {type(judge_labels).__name__}.")
if len(judge_preds) == 0:
    raise ValueError("FAILED: judge_preds is empty. Check that run_llm_judge returns results.")
if len(judge_preds) != len(judge_labels):
    raise ValueError(
        f"FAILED: len(judge_preds)={len(judge_preds)} != len(judge_labels)={len(judge_labels)}."
    )
if not all(p in (0, 1) for p in judge_preds):
    raise ValueError("FAILED: judge_preds must contain only 0 or 1.")
_required_judge_acc = 0.6
if judge_acc < _required_judge_acc:
    raise ValueError(
        f"FAILED: LLM judge accuracy is too low.\n"
        f"Your accuracy:    {judge_acc:.4f}\n"
        f"Required minimum: {_required_judge_acc:.4f}\n"
        f"Improve your prompt in build_judge_prompt."
    )
print(f"Part 1c - run_llm_judge passed. (accuracy={judge_acc:.4f}, n={len(judge_preds)})")

Part 1c - run_llm_judge passed. (accuracy=0.8261, n=23)


---
## Part 2 - Reward Model Training (50 pts)

We fine-tune `microsoft/deberta-v3-small` as a reward model, which produces a single scalar reward $r(x, y)$ for each (prompt $x$, response $y$) pair.

**Training objective -- Bradley-Terry loss:**

$$L = -\log \sigma\!\left(r(x, y_w) - r(x, y_l)\right)$$

where $y_w$ is the preferred response and $y_l$ is the rejected response.

**Expected training time:** ~25 minutes on a Colab T4.

In [14]:
RM_BASE_MODEL = "microsoft/deberta-v3-small"

### 2a - Dataset (10 pts)

Implement a `torch.utils.data.Dataset` that tokenises pairwise preference examples.

**Steps for `__init__`:**
1. Iterate over each example in `data`. Each example is a dict with keys `prompt`, `response_a`, `response_b`, and `label`.
2. Determine which response is **chosen** (preferred) and which is **rejected**:
   - `label == 0` → `response_a` is chosen, `response_b` is rejected
   - `label == 1` → `response_b` is chosen, `response_a` is rejected
3. Tokenize each `(prompt, response)` pair as a **sentence pair**:
   ```python
   tokenizer(prompt, response, truncation=True, max_length=max_length)
   ```
4. Store the tokenized chosen and rejected encodings (e.g. in two lists `self.chosen`, `self.rejected`).

**`__len__`:** return the number of examples.

**`__getitem__(idx)`:** return a dict with four `torch.long` tensors:
```python
{
    "chosen_input_ids":        torch.tensor(..., dtype=torch.long),
    "chosen_attention_mask":   torch.tensor(..., dtype=torch.long),
    "rejected_input_ids":      torch.tensor(..., dtype=torch.long),
    "rejected_attention_mask": torch.tensor(..., dtype=torch.long),
}
```

In [15]:
class RewardDataset(torch.utils.data.Dataset):
    """
    Tokenised pairwise preference dataset.

    __getitem__ returns a dict with keys:
        chosen_input_ids, chosen_attention_mask,
        rejected_input_ids, rejected_attention_mask
    """

    def __init__(self, data: list, tokenizer, max_length: int = 512):
        self.chosen = []
        self.rejected = []
        for ex in data:
            # label == 0 -> response_a chosen; label == 1 -> response_b chosen
            if ex["label"] == 0:
                chosen_resp, rejected_resp = ex["response_a"], ex["response_b"]
            else:
                chosen_resp, rejected_resp = ex["response_b"], ex["response_a"]

            self.chosen.append(
                tokenizer(ex["prompt"], chosen_resp,
                          truncation=True, max_length=max_length)
            )
            self.rejected.append(
                tokenizer(ex["prompt"], rejected_resp,
                          truncation=True, max_length=max_length)
            )

    def __len__(self):
        return len(self.chosen)

    def __getitem__(self, idx):
        c = self.chosen[idx]
        r = self.rejected[idx]
        return {
            "chosen_input_ids":        torch.tensor(c["input_ids"], dtype=torch.long),
            "chosen_attention_mask":   torch.tensor(c["attention_mask"], dtype=torch.long),
            "rejected_input_ids":      torch.tensor(r["input_ids"], dtype=torch.long),
            "rejected_attention_mask": torch.tensor(r["attention_mask"], dtype=torch.long),
        }

In [16]:
# =====================================================
# DO NOT MODIFY
# =====================================================

from torch.utils.data import Dataset as _Dataset
from transformers import AutoTokenizer

_tok_check = AutoTokenizer.from_pretrained(RM_BASE_MODEL)
_ds_check = RewardDataset(train_data[:4], _tok_check, max_length=64)

if not isinstance(_ds_check, _Dataset):
    raise ValueError("FAILED: RewardDataset must inherit from torch.utils.data.Dataset.")
if len(_ds_check) != 4:
    raise ValueError(f"FAILED: __len__ returned {len(_ds_check)}, expected 4.")
_item_check = _ds_check[0]
_required_keys = {"chosen_input_ids", "chosen_attention_mask", "rejected_input_ids", "rejected_attention_mask"}
if not _required_keys.issubset(set(_item_check.keys())):
    raise ValueError(
        f"FAILED: __getitem__ missing keys.\n"
        f"  Got:      {set(_item_check.keys())}\n"
        f"  Expected: {_required_keys}"
    )
if not isinstance(_item_check["chosen_input_ids"], torch.Tensor):
    raise ValueError("FAILED: chosen_input_ids must be a torch.Tensor.")
if _item_check["chosen_input_ids"].shape != _item_check["chosen_attention_mask"].shape:
    raise ValueError("FAILED: chosen_input_ids and chosen_attention_mask must have the same shape.")
del _tok_check, _ds_check, _item_check
print("Part 2a - RewardDataset structure checks passed.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Part 2a - RewardDataset structure checks passed.


### 2b - Bradley-Terry Loss (5 pts)

In [17]:
def bradley_terry_loss(
    chosen_rewards: torch.Tensor,
    rejected_rewards: torch.Tensor,
) -> torch.Tensor:
    """
    Bradley-Terry pairwise ranking loss.

    Args:
        chosen_rewards:   shape (B,)
        rejected_rewards: shape (B,)

    Returns:
        Scalar loss tensor.
    """
    # L = -mean( log sigmoid( r_chosen - r_rejected ) )
    # Using logsigmoid for numerical stability.
    return -torch.nn.functional.logsigmoid(chosen_rewards - rejected_rewards).mean()

In [18]:
# =====================================================
# DO NOT MODIFY
# =====================================================

import math as _math
_chosen_r = torch.tensor([1.0, 2.0, 3.0])
_rejected_r = torch.tensor([0.0, 0.0, 0.0])
_loss_val = bradley_terry_loss(_chosen_r, _rejected_r)
if not isinstance(_loss_val, torch.Tensor):
    raise ValueError(f"FAILED: bradley_terry_loss must return a torch.Tensor, got {type(_loss_val).__name__}.")
if _loss_val.ndim != 0:
    raise ValueError(f"FAILED: bradley_terry_loss must return a scalar tensor, got shape {tuple(_loss_val.shape)}.")
if _loss_val.item() < 0:
    raise ValueError(f"FAILED: loss must be non-negative, got {_loss_val.item():.6f}.")
# When chosen == rejected, loss = -log(sigma(0)) = log(2) ≈ 0.6931
_equal_r = torch.zeros(4)
_loss_equal = bradley_terry_loss(_equal_r, _equal_r)
_expected_equal = _math.log(2)
if abs(_loss_equal.item() - _expected_equal) > 1e-4:
    raise ValueError(
        f"FAILED: When chosen == rejected, loss should be {_expected_equal:.6f} (log 2), "
        f"got {_loss_equal.item():.6f}."
    )
print("Part 2b - bradley_terry_loss correctness checks passed.")

Part 2b - bradley_terry_loss correctness checks passed.


### 2c - Build the model (5 pts)

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


def build_reward_model(base_model_name: str = RM_BASE_MODEL):
    """
    Load a pretrained model with a single scalar output head.

    Returns:
        model, tokenizer
    """
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    # num_labels=1 -> the classification head outputs a single scalar reward.
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name, num_labels=1
    )

    return model, tokenizer

In [20]:
# =====================================================
# DO NOT MODIFY
# =====================================================

from transformers import PreTrainedTokenizerBase as _PreTrainedTokenizerBase
_rm_check, _tok_rm_check = build_reward_model()
if not isinstance(_rm_check, torch.nn.Module):
    raise ValueError(f"FAILED: build_reward_model must return an nn.Module as first value, got {type(_rm_check).__name__}.")
if not isinstance(_tok_rm_check, _PreTrainedTokenizerBase):
    raise ValueError(f"FAILED: build_reward_model must return a tokenizer as second value, got {type(_tok_rm_check).__name__}.")
if _rm_check.config.num_labels != 1:
    raise ValueError(
        f"FAILED: model must be built with num_labels=1 for scalar reward output, "
        f"got num_labels={_rm_check.config.num_labels}."
    )
del _rm_check, _tok_rm_check
print("Part 2c - build_reward_model checks passed.")

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

Part 2c - build_reward_model checks passed.


### 2d - Training loop (30 pts)

**Tips:**
- Use `transformers.get_linear_schedule_with_warmup` with 10 % warmup steps.
- Forward chosen and rejected separately; each call returns `.logits` of shape `(B, 1)` - squeeze to `(B,)` before passing to the loss.

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from transformers import get_linear_schedule_with_warmup

def train_reward_model(
    train_data: list,
    num_epochs: int = 2,
    batch_size: int = 8,
    lr: float = 2e-5,
    max_length: int = 512,
    save_path: str = "reward_model",
) -> torch.nn.Module:
    """
    Fine-tune the reward model on preference pairs.

    Steps:
      1. build_reward_model() -> model, tokenizer
      2. RewardDataset + DataLoader
      3. AdamW optimizer + linear LR schedule (10% warmup)
      4. Training loop:
           a. Forward chosen  -> squeeze logits -> chosen_rewards
           b. Forward rejected -> squeeze logits -> rejected_rewards
           c. loss = bradley_terry_loss(chosen_rewards, rejected_rewards)
           d. back propagation
      5. return model
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    # 1. Model + tokenizer
    model, tokenizer = build_reward_model()
    model.to(device)

    # 2. Dataset + DataLoader. Examples have variable length, so we pad each
    #    batch dynamically in a collate function.
    dataset = RewardDataset(train_data, tokenizer, max_length=max_length)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    def collate_fn(batch):
        def _pad(ids_key, mask_key):
            ids = pad_sequence([b[ids_key] for b in batch],
                               batch_first=True, padding_value=pad_id)
            mask = pad_sequence([b[mask_key] for b in batch],
                                batch_first=True, padding_value=0)
            return ids, mask

        c_ids, c_mask = _pad("chosen_input_ids", "chosen_attention_mask")
        r_ids, r_mask = _pad("rejected_input_ids", "rejected_attention_mask")
        return {
            "chosen_input_ids": c_ids, "chosen_attention_mask": c_mask,
            "rejected_input_ids": r_ids, "rejected_attention_mask": r_mask,
        }

    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
    )

    # 3. Optimizer + linear schedule with 10% warmup
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = max(1, len(loader) * num_epochs)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # 4. Training loop
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        n_batches = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch + 1}/{num_epochs}")
        for batch in pbar:
            optimizer.zero_grad()

            chosen_rewards = model(
                input_ids=batch["chosen_input_ids"].to(device),
                attention_mask=batch["chosen_attention_mask"].to(device),
            ).logits.squeeze(-1)

            rejected_rewards = model(
                input_ids=batch["rejected_input_ids"].to(device),
                attention_mask=batch["rejected_attention_mask"].to(device),
            ).logits.squeeze(-1)

            loss = bradley_terry_loss(chosen_rewards, rejected_rewards)
            loss.backward()
            for p in model.parameters():
                if p.grad is not None:
                    torch.nan_to_num_(p.grad, nan=0.0, posinf=0.0, neginf=0.0)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=running_loss / n_batches)

        print(f"Epoch {epoch + 1}: avg loss = {running_loss / max(1, n_batches):.4f}")

    return model

In [22]:
# =====================================================
# DO NOT MODIFY
# =====================================================
_n_eval = 5
model = train_reward_model(train_data[:_n_eval])

from transformers import AutoTokenizer as _AutoTokenizer

# 1. Type and architecture checks
if not isinstance(model, torch.nn.Module):
    raise ValueError(f"FAILED: train_reward_model must return an nn.Module, got {type(model).__name__}.")
if next(model.parameters(), None) is None:
    raise ValueError("FAILED: The returned model has no parameters.")
if not hasattr(model, "config") or model.config.num_labels != 1:
    raise ValueError(
        f"FAILED: model must have num_labels=1 for scalar reward output, "
        f"got num_labels={getattr(getattr(model, 'config', None), 'num_labels', 'N/A')}."
    )
print("Part 2c - train_reward_model returned a valid trained model.")

# 2. Forward pass produces logits of shape (B, 1)
_device_2d = next(model.parameters()).device
_tok_2d = _AutoTokenizer.from_pretrained(RM_BASE_MODEL)
_probe_sample = train_data[:4]
_enc_probe = _tok_2d(
    [ex["prompt"] for ex in _probe_sample],
    [ex["response_a"] for ex in _probe_sample],
    truncation=True, max_length=128, padding=True, return_tensors="pt",
)
model.eval()
with torch.no_grad():
    _probe_logits = model(
        input_ids=_enc_probe["input_ids"].to(_device_2d),
        attention_mask=_enc_probe["attention_mask"].to(_device_2d),
    ).logits
if tuple(_probe_logits.shape) != (4, 1):
    raise ValueError(
        f"FAILED: model logits should have shape (batch_size, 1), got {tuple(_probe_logits.shape)}."
    )

# 3. Pairwise accuracy on a small training sample
# A correctly trained model should rank chosen > rejected well above random.
_eval_data = train_data[:_n_eval]
_prompts_2d  = [ex["prompt"]     for ex in _eval_data]
_chosen_2d   = [ex["response_a"] if ex["label"] == 0 else ex["response_b"] for ex in _eval_data]
_rejected_2d = [ex["response_b"] if ex["label"] == 0 else ex["response_a"] for ex in _eval_data]

_enc_c = _tok_2d(_prompts_2d, _chosen_2d,   truncation=True, max_length=256, padding=True, return_tensors="pt")
_enc_r = _tok_2d(_prompts_2d, _rejected_2d, truncation=True, max_length=256, padding=True, return_tensors="pt")

with torch.no_grad():
    _rewards_c = model(input_ids=_enc_c["input_ids"].to(_device_2d), attention_mask=_enc_c["attention_mask"].to(_device_2d)).logits.squeeze(-1).cpu()
    _rewards_r = model(input_ids=_enc_r["input_ids"].to(_device_2d), attention_mask=_enc_r["attention_mask"].to(_device_2d)).logits.squeeze(-1).cpu()

_pairwise_acc_2d = (_rewards_c > _rewards_r).float().mean().item()
_required_pairwise_acc = 0.55
if _pairwise_acc_2d < _required_pairwise_acc:
    raise ValueError(
        f"FAILED: Trained model does not assign higher reward to chosen responses.\n"
        f"Pairwise accuracy on {_n_eval} training samples: {_pairwise_acc_2d:.4f}\n"
        f"Required minimum: {_required_pairwise_acc:.4f}\n"
        f"Ensure you use bradley_terry_loss with the correct chosen/rejected assignment."
    )
del _tok_2d, _enc_probe, _enc_c, _enc_r
print(f"Part 2d - train_reward_model checks passed. (train pairwise acc={_pairwise_acc_2d:.4f})")

Training on: cuda


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

Epoch 1/2:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1: avg loss = 0.6973


Epoch 2/2:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2: avg loss = nan
Part 2c - train_reward_model returned a valid trained model.


ValueError: FAILED: Trained model does not assign higher reward to chosen responses.
Pairwise accuracy on 5 training samples: 0.0000
Required minimum: 0.5500
Ensure you use bradley_terry_loss with the correct chosen/rejected assignment.

In [ ]:
model = train_reward_model(train_data)

---
## Part 3 - Comparison & Analysis (25 pts)

### 3a - Reward model inference (15 pts)

Score both responses for every test example and predict the higher-reward one as preferred.

In [ ]:
def predict_with_reward_model(
    model: torch.nn.Module,
    tokenizer,
    test_data: list,
    batch_size: int = 16,
    max_length: int = 512,
) -> tuple:
    """
    Batched inference on the full test set.

    For each example:
      - Score (prompt, response_a) and (prompt, response_b).
      - Predict 0 if reward_a > reward_b, else 1.

    Returns:
        predictions (list[int]), labels (list[int])
    """
    device = next(model.parameters()).device
    model.eval()

    predictions, labels = [], []

    def _score(prompts, responses):
        enc = tokenizer(
            prompts, responses,
            truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        )
        with torch.no_grad():
            logits = model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
            ).logits
        return logits.squeeze(-1).cpu()

    for i in range(0, len(test_data), batch_size):
        batch = test_data[i:i + batch_size]
        prompts = [ex["prompt"] for ex in batch]
        resp_a = [ex["response_a"] for ex in batch]
        resp_b = [ex["response_b"] for ex in batch]

        rewards_a = _score(prompts, resp_a)
        rewards_b = _score(prompts, resp_b)

        # Predict 0 (A preferred) if reward_a > reward_b, else 1 (B preferred).
        batch_preds = (rewards_b > rewards_a).long().tolist()
        predictions.extend(batch_preds)
        labels.extend([ex["label"] for ex in batch])

    return predictions, labels

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(RM_BASE_MODEL)
rm_preds, rm_labels = predict_with_reward_model(model, tokenizer, test_data)
rm_acc = compute_accuracy(rm_preds, rm_labels)
print(f"Reward Model Accuracy: {rm_acc:.4f}  ({len(rm_preds)} examples)")

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

if not isinstance(rm_preds, list):
    raise ValueError(f"FAILED: rm_preds must be a list, got {type(rm_preds).__name__}.")
if not isinstance(rm_labels, list):
    raise ValueError(f"FAILED: rm_labels must be a list, got {type(rm_labels).__name__}.")
if len(rm_preds) == 0:
    raise ValueError("FAILED: rm_preds is empty. Check predict_with_reward_model.")
if len(rm_preds) != len(rm_labels):
    raise ValueError(
        f"FAILED: len(rm_preds)={len(rm_preds)} != len(rm_labels)={len(rm_labels)}."
    )
if not all(p in (0, 1) for p in rm_preds):
    raise ValueError("FAILED: rm_preds must contain only 0 or 1.")
_required_rm_acc = 0.58
if rm_acc < _required_rm_acc:
    raise ValueError(
        f"FAILED: Reward model accuracy is too low.\n"
        f"Your accuracy:    {rm_acc:.4f}\n"
        f"Required minimum: {_required_rm_acc:.4f}\n"
        f"Try training for more epochs or adjusting the learning rate."
    )
print(f"Part 3a - predict_with_reward_model passed. (accuracy={rm_acc:.4f}, n={len(rm_preds)})")

### 3b - Comparison table & written analysis (10 pts)

Your output must include:
- Accuracy of both systems
- Fraction predicted as A / B for each system (position-bias check)
- **2–3 sentences of analysis** (in the Markdown cell below)

In [ ]:
def run_comparison(
    judge_preds: list,
    judge_labels: list,
    rm_preds: list,
    rm_labels: list,
) -> None:
    """
    Print a formatted comparison table.

    Report for both systems:
      - Accuracy
      - Fraction predicted A  (position-bias check)
      - Fraction predicted B
    """
    def _stats(preds, labels):
        acc = compute_accuracy(preds, labels)
        n = len(preds)
        frac_a = sum(1 for p in preds if p == 0) / n  # predicted A
        frac_b = sum(1 for p in preds if p == 1) / n  # predicted B
        return acc, frac_a, frac_b, n

    j_acc, j_a, j_b, j_n = _stats(judge_preds, judge_labels)
    r_acc, r_a, r_b, r_n = _stats(rm_preds, rm_labels)

    header = f"{'System':<16}{'N':>6}{'Accuracy':>12}{'Frac A':>10}{'Frac B':>10}"
    print(header)
    print("-" * len(header))
    print(f"{'LLM Judge':<16}{j_n:>6}{j_acc:>12.4f}{j_a:>10.4f}{j_b:>10.4f}")
    print(f"{'Reward Model':<16}{r_n:>6}{r_acc:>12.4f}{r_a:>10.4f}{r_b:>10.4f}")

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

# Verify run_comparison completes without error on the collected predictions
try:
    run_comparison(judge_preds, judge_labels, rm_preds, rm_labels)
except Exception as _e:
    raise ValueError(f"FAILED: run_comparison raised an exception: {_e}")
print("Part 3b - run_comparison completed successfully.")

#### Analysis

*(Replace this cell with your 2–3 sentence analysis. Discuss where each approach wins or loses and what you think explains the difference.)*